# 06 - Advanced Keras Custom Objects

This notebook collects the advanced Keras constructs requested in the assignment.

## Covered items
1. custom learning-rate scheduler
2. custom dropout
3. custom normalization
4. TensorBoard
5. custom loss function
6. custom activation / initializer / regularizer / kernel constraint
7. custom metric
8. custom layers
9. custom model
10. custom optimizer
11. custom training loop

To keep the notebook coherent, most custom objects are demonstrated on a synthetic regression task, then the custom training loop is shown on a small Fashion-MNIST classification task.


In [ ]:
# Colab setup
%pip -q install -U tensorflow scikit-learn tensorboard pandas matplotlib


In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("TensorFlow:", tf.__version__)


## Regression dataset


In [ ]:
X, y = make_regression(
    n_samples=6000,
    n_features=20,
    n_informative=14,
    noise=12.0,
    random_state=SEED,
)

y = y.astype("float32").reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X.astype("float32"), y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

X_train, X_val = X_train[:-800], X_train[-800:]
y_train, y_val = y_train[:-800], y_train[-800:]

X_train.shape, X_val.shape, X_test.shape


## Custom building blocks


In [ ]:
def scaled_leaky_relu(x):
    return 1.15 * tf.nn.leaky_relu(x, alpha=0.15)

class MyGlorotInitializer(keras.initializers.Initializer):
    def __call__(self, shape, dtype=None):
        limit = tf.sqrt(6.0 / tf.cast(shape[0] + shape[1], tf.float32))
        return tf.random.uniform(shape, -limit, limit, dtype=dtype or tf.float32)

class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self, strength=1e-4):
        self.strength = strength

    def __call__(self, x):
        return self.strength * tf.reduce_sum(tf.abs(x))

    def get_config(self):
        return {"strength": self.strength}

class PositiveWeights(keras.constraints.Constraint):
    def __call__(self, w):
        return tf.nn.relu(w)

class CustomMCDropout(layers.Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

class FeatureMaxNorm(layers.Layer):
    def __init__(self, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def call(self, inputs):
        scale = tf.reduce_max(tf.abs(inputs), axis=-1, keepdims=True)
        return inputs / (scale + self.eps)

    def get_config(self):
        return {"eps": self.eps, **super().get_config()}


In [ ]:
class SmoothHuberLoss(keras.losses.Loss):
    def __init__(self, delta=1.0, name="smooth_huber_loss"):
        super().__init__(name=name)
        self.delta = delta

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        abs_error = tf.abs(error)
        quadratic = tf.minimum(abs_error, self.delta)
        linear = abs_error - quadratic
        return tf.reduce_mean(0.5 * tf.square(quadratic) + self.delta * linear)

    def get_config(self):
        return {"delta": self.delta}

class HuberMetric(keras.metrics.Metric):
    def __init__(self, delta=1.0, name="huber_metric", **kwargs):
        super().__init__(name=name, **kwargs)
        self.delta = delta
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        error = tf.abs(y_true - y_pred)
        quadratic = tf.minimum(error, self.delta)
        linear = error - quadratic
        value = tf.reduce_mean(0.5 * tf.square(quadratic) + self.delta * linear)
        self.total.assign_add(value)
        self.count.assign_add(1.0)

    def result(self):
        return self.total / tf.maximum(self.count, 1.0)

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)


In [ ]:
class ExponentialLayer(layers.Layer):
    def call(self, inputs):
        return tf.exp(tf.clip_by_value(inputs, -5.0, 5.0))

class AddGaussianNoise(layers.Layer):
    def __init__(self, stddev=0.05, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(tf.shape(inputs), stddev=self.stddev)
            return inputs + noise
        return inputs

    def get_config(self):
        return {"stddev": self.stddev, **super().get_config()}

class LayerNormalizationLite(layers.Layer):
    def __init__(self, eps=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        dim = input_shape[-1]
        self.gamma = self.add_weight(shape=(dim,), initializer="ones", trainable=True)
        self.beta = self.add_weight(shape=(dim,), initializer="zeros", trainable=True)

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        var = tf.reduce_mean(tf.square(inputs - mean), axis=-1, keepdims=True)
        norm = (inputs - mean) / tf.sqrt(var + self.eps)
        return norm * self.gamma + self.beta

class MyDense(layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer=MyGlorotInitializer(),
            regularizer=MyL1Regularizer(1e-5),
            constraint=PositiveWeights(),
            trainable=True,
        )
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)

    def call(self, inputs):
        z = tf.matmul(inputs, self.w) + self.b
        return self.activation(z) if self.activation is not None else z


In [ ]:
class ResidualBlock(layers.Layer):
    def __init__(self, units, dropout=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dropout = dropout
        self.dense1 = MyDense(units, activation=scaled_leaky_relu)
        self.norm1 = LayerNormalizationLite()
        self.noise = AddGaussianNoise(0.03)
        self.drop = CustomMCDropout(dropout)
        self.dense2 = MyDense(units, activation=None)
        self.norm2 = LayerNormalizationLite()

    def build(self, input_shape):
        if input_shape[-1] != self.units:
            self.proj = layers.Dense(self.units)
        else:
            self.proj = None

    def call(self, inputs, training=None):
        shortcut = self.proj(inputs) if self.proj is not None else inputs
        x = self.dense1(inputs)
        x = self.norm1(x)
        x = self.noise(x, training=training)
        x = self.drop(x, training=training)
        x = self.dense2(x)
        x = self.norm2(x)
        return tf.nn.relu(x + shortcut)

class ResidualRegressor(keras.Model):
    def __init__(self, units=64):
        super().__init__()
        self.input_proj = layers.Dense(
            units,
            activation=scaled_leaky_relu,
            kernel_initializer=MyGlorotInitializer(),
            kernel_regularizer=MyL1Regularizer(1e-5),
        )
        self.feature_norm = FeatureMaxNorm()
        self.block1 = ResidualBlock(units)
        self.block2 = ResidualBlock(units)
        self.out = layers.Dense(1)

    def call(self, inputs, training=None):
        x = self.input_proj(inputs)
        x = self.feature_norm(x)
        x = self.block1(x, training=training)
        x = self.block2(x, training=training)
        return self.out(x)


## Custom optimizer and custom learning-rate scheduler


In [ ]:
class SimpleMomentumOptimizer(keras.optimizers.Optimizer):
    def __init__(self, learning_rate=0.01, momentum=0.9, name="simple_momentum", **kwargs):
        super().__init__(learning_rate=learning_rate, name=name, **kwargs)
        self.momentum = momentum

    def build(self, variables):
        super().build(variables)
        self.momentums = []
        for var in variables:
            self.momentums.append(self.add_variable_from_reference(reference_variable=var, name="momentum"))

    def update_step(self, gradient, variable, learning_rate=None):
        lr = tf.cast(learning_rate if learning_rate is not None else self.learning_rate, variable.dtype)
        grad = tf.cast(gradient, variable.dtype)
        idx = self._get_variable_index(variable)
        m = self.momentums[idx]
        self.assign(m, self.momentum * m - lr * grad)
        self.assign_add(variable, m)

    def get_config(self):
        config = super().get_config()
        config.update({"momentum": self.momentum})
        return config

class OneCycleStyleScheduler(keras.callbacks.Callback):
    def __init__(self, max_lr=3e-3, start_lr=3e-4, final_lr=1e-4, warmup_epochs=3, total_epochs=15):
        super().__init__()
        self.max_lr = max_lr
        self.start_lr = start_lr
        self.final_lr = final_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.history = []

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.max_lr - self.start_lr) * (epoch / max(1, self.warmup_epochs))
        else:
            progress = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
            lr = self.max_lr - (self.max_lr - self.final_lr) * progress

        self.model.optimizer.learning_rate.assign(lr)
        self.history.append(lr)
        print(f"Epoch {epoch + 1}: learning_rate={lr:.6f}")


## Compile and train the custom model with TensorBoard


In [ ]:
reg_model = ResidualRegressor(units=64)
reg_model.compile(
    optimizer=SimpleMomentumOptimizer(learning_rate=3e-4, momentum=0.9),
    loss=SmoothHuberLoss(delta=1.2),
    metrics=[keras.metrics.MeanAbsoluteError(name="mae"), HuberMetric(delta=1.2)],
)

log_dir = os.path.join("logs", "part2_keras_custom")
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_mae", patience=4, restore_best_weights=True),
    keras.callbacks.TensorBoard(log_dir=log_dir),
    OneCycleStyleScheduler(max_lr=2e-3, start_lr=3e-4, final_lr=1e-4, warmup_epochs=3, total_epochs=15),
]

history = reg_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
reg_eval = reg_model.evaluate(X_test, y_test, verbose=0)
pd.DataFrame(
    [{"metric": name, "value": value} for name, value in zip(reg_model.metrics_names, reg_eval)]
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["mae"], label="train_mae")
plt.plot(history.history["val_mae"], label="val_mae")
plt.title("Custom Keras model training")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.show()


## Monte Carlo dropout uncertainty on the regressor


In [ ]:
sample_X = X_test[:32]
mc_preds = []
for _ in range(25):
    mc_preds.append(reg_model(sample_X, training=True).numpy().squeeze())
mc_preds = np.stack(mc_preds)

mean_pred = mc_preds.mean(axis=0)
std_pred = mc_preds.std(axis=0)

pd.DataFrame({
    "prediction_mean": mean_pred[:10],
    "prediction_std": std_pred[:10],
    "target": y_test[:10].squeeze(),
})


## Custom training loop example

Now we switch to a small Fashion-MNIST classification task and show an explicit `GradientTape` training loop.


In [ ]:
(x_train_fm, y_train_fm), (x_test_fm, y_test_fm) = keras.datasets.fashion_mnist.load_data()
x_train_fm = x_train_fm[:10000].astype("float32") / 255.0
y_train_fm = y_train_fm[:10000]
x_test_fm = x_test_fm[:2000].astype("float32") / 255.0
y_test_fm = y_test_fm[:2000]

train_ds = tf.data.Dataset.from_tensor_slices((x_train_fm, y_train_fm)).shuffle(4000).batch(128)
test_ds = tf.data.Dataset.from_tensor_slices((x_test_fm, y_test_fm)).batch(256)

class SmallClassifier(keras.Model):
    def __init__(self):
        super().__init__()
        self.flatten = layers.Flatten()
        self.d1 = layers.Dense(128, activation=scaled_leaky_relu)
        self.drop = CustomMCDropout(0.25)
        self.out = layers.Dense(10)

    def call(self, x, training=None):
        x = self.flatten(x)
        x = self.d1(x)
        x = self.drop(x, training=training)
        return self.out(x)

classifier = SmallClassifier()
optimizer = keras.optimizers.Adam(1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
train_acc = keras.metrics.SparseCategoricalAccuracy()
test_acc = keras.metrics.SparseCategoricalAccuracy()


In [ ]:
epochs = 5
for epoch in range(epochs):
    train_acc.reset_state()
    test_acc.reset_state()
    train_losses = []

    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            logits = classifier(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
            loss += tf.add_n(classifier.losses) if classifier.losses else 0.0

        grads = tape.gradient(loss, classifier.trainable_variables)
        optimizer.apply_gradients(zip(grads, classifier.trainable_variables))
        train_acc.update_state(y_batch, logits)
        train_losses.append(float(loss))

    for x_batch, y_batch in test_ds:
        test_logits = classifier(x_batch, training=False)
        test_acc.update_state(y_batch, test_logits)

    print(
        f"epoch={epoch+1} "
        f"train_loss={np.mean(train_losses):.4f} "
        f"train_acc={train_acc.result().numpy():.4f} "
        f"test_acc={test_acc.result().numpy():.4f}"
    )


In [ ]:
print("TensorBoard log root:", os.path.abspath(log_dir))
# In Colab:
# %load_ext tensorboard
# %tensorboard --logdir logs


## Wrap-up

This notebook intentionally packs many advanced Keras features into one place:

- subclassed layers and models
- custom optimizer
- custom LR schedule callback
- custom loss / metric
- custom activation, initializer, regularizer, constraint
- custom training loop with `tf.GradientTape`

For the video walkthrough, explain the ideas in clusters instead of reading every line mechanically. Otherwise the recording will become a heroic but unnecessary endurance sport.
